In [1]:
import numpy as np
import os
import json
from zipfile import ZipFile
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow import keras
from keras import layers, models

2025-03-30 20:07:13.942459: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-30 20:07:13.956744: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-30 20:07:13.974062: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-30 20:07:13.980004: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-30 20:07:13.993673: I tensorflow/core/platform/cpu_feature_guar

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential,load_model
from keras import layers
from keras.layers import Dense, Activation, Flatten, Dropout, BatchNormalization,Embedding,TimeDistributed
from keras.layers import Conv2D, MaxPooling2D, ReLU, LSTM,Bidirectional,Attention,Concatenate,concatenate
from keras import regularizers, optimizers,losses
from keras.layers import DepthwiseConv2D,Add, ReLU, GlobalAveragePooling2D, GlobalMaxPooling2D,MultiHeadAttention
from keras.layers import Activation,ActivityRegularization, AvgPool2D, LeakyReLU, Conv2DTranspose
from keras.metrics import Accuracy,Recall,Precision,AUC,TruePositives,TrueNegatives,FalseNegatives,FalsePositives, SpecificityAtSensitivity,SensitivityAtSpecificity
from keras.utils import plot_model
from keras.preprocessing import image
from keras.utils import to_categorical
import numpy as np
import matplotlib
import sklearn
import matplotlib.pyplot as plt
import time
import os
import sklearn.metrics as m
from glob import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import tree
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

In [3]:
import os
import numpy as np
from glob import glob
from keras.preprocessing import image
from sklearn.model_selection import train_test_split

input_dir = "/home/gourib/LIDC-IDRI-slices"
img_shape = (128,128)  # Adjust according to your requirement
Thres = Thres = 5000/255

images = []
masks = []
y = []

# Step 1: Get all patient IDs
patients = sorted(os.listdir(input_dir))

# Step 2: Split patients into train and test sets
train_patients, test_patients = train_test_split(patients, train_size=0.8, random_state=25)

def load_patient_data(patient_list):
    """Load images and masks for the given list of patients"""
    images, masks, labels = [], [], []
    
    for patient in patient_list:
        patient_path = os.path.join(input_dir, patient)
        
        for nodule in os.listdir(patient_path):
            mask0, mask1, mask2, mask3 = [], [], [], []

            # Load images
            for filename in glob(f"{patient_path}/{nodule}/images/*.png"):
                img = image.load_img(filename, target_size=img_shape)
                images.append(np.asarray(img))

            # Load masks
            for filename in glob(f"{patient_path}/{nodule}/mask-0/*.png"):
                img = image.load_img(filename, target_size=img_shape)
                mask0.append(np.asarray(img))
            for filename in glob(f"{patient_path}/{nodule}/mask-1/*.png"):
                img = image.load_img(filename, target_size=img_shape)
                mask1.append(np.asarray(img))
            for filename in glob(f"{patient_path}/{nodule}/mask-2/*.png"):
                img = image.load_img(filename, target_size=img_shape)
                mask2.append(np.asarray(img))
            for filename in glob(f"{patient_path}/{nodule}/mask-3/*.png"):
                img = image.load_img(filename, target_size=img_shape)
                mask3.append(np.asarray(img))

            # Determine final mask selection
            for i in range(len(mask0)):
                white_sum = np.array([mask0[i].sum(), mask1[i].sum(), mask2[i].sum(), mask3[i].sum()])
                cnt = sum(mask.sum() > Thres for mask in [mask0[i], mask1[i], mask2[i], mask3[i]])

                if cnt > 2:
                    labels.append(1)
                    masks.append([mask0[i], mask1[i], mask2[i], mask3[i]][white_sum.argmax()])
                else:
                    labels.append(0)
                    masks.append([mask0[i], mask1[i], mask2[i], mask3[i]][white_sum.argmin()])
    
    return images, masks, labels
    

# Step 3: Load training and testing data based on patient-wise split
x_train, mask_train, y_train = load_patient_data(train_patients)
x_test, mask_test, y_test = load_patient_data(test_patients)

# Convert to NumPy arrays
x_train, x_test = np.array(x_train)/255, np.array(x_test)/255
mask_train, mask_test = np.array(mask_train)/255, np.array(mask_test)/255
y_train, y_test = np.array(y_train), np.array(y_test)

# Print shapes
print(x_train.shape, x_test.shape)
print(mask_train.shape, mask_test.shape)

(12376, 128, 128, 3) (3172, 128, 128, 3)
(12376, 128, 128, 3) (3172, 128, 128, 3)


In [4]:
import tensorflow as tf

# Download InceptionV3 weights
inception_weights_path = tf.keras.utils.get_file(
    'inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5',
    'https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5'
)

# Download VGG16 weights
vgg16_weights_path = tf.keras.utils.get_file(
    'vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5',
    'https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5'
)

print("InceptionV3 weights saved at:", inception_weights_path)
print("VGG16 weights saved at:", vgg16_weights_path)


InceptionV3 weights saved at: /home/gourib/.keras/datasets/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5
VGG16 weights saved at: /home/gourib/.keras/datasets/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5


In [5]:
import tensorflow as tf
from keras.applications import VGG16, InceptionV3
from keras.layers import GlobalAveragePooling2D, Dense, Dropout, Concatenate
from keras.models import Model
from keras.optimizers import Adam
vgg16_weights_path = "/home/gourib/.keras/datasets/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
inception_weights_path = "/home/gourib/.keras/datasets/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
# Load pre-trained models with local weights
vgg16_base = VGG16(weights=vgg16_weights_path, include_top=False, input_shape=(128,128, 3))
inception_base = InceptionV3(weights=inception_weights_path, include_top=False, input_shape=(128,128, 3))

# Freezing 25% of layers in both models
for layer in vgg16_base.layers[:int(len(vgg16_base.layers) * 0.3)]:
    layer.trainable = False
for layer in inception_base.layers[:int(len(inception_base.layers) * 0.3)]:
    layer.trainable = False

# Combine outputs
vgg16_output = GlobalAveragePooling2D()(vgg16_base.output)
inception_output = GlobalAveragePooling2D()(inception_base.output)
combined_features = Concatenate()([vgg16_output, inception_output])
x = Dense(256, activation='relu')(combined_features)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

# Build and compile the ensemble model

2025-03-30 20:07:42.518278: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0


In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.metrics import Recall, Precision, AUC, TruePositives, TrueNegatives, FalseNegatives, FalsePositives

ensemble_model = Model(inputs=[vgg16_base.input, inception_base.input], outputs=output)

from keras.callbacks import EarlyStopping
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
# keras.optimizers.Adam()
ensemble_model.compile (optimizer=optimizer ,  loss=keras.losses.BinaryCrossentropy() , 
               metrics=['acc',Recall(),Precision(),AUC(),TruePositives(),TrueNegatives(),FalseNegatives(),FalsePositives()])
early_stopping = EarlyStopping(
    monitor='val_acc', 
    patience=15, 
    min_delta=0.001, 
    mode='max',
    restore_best_weights=True
)


In [7]:
history = ensemble_model.fit([x_train,x_train], y_train, epochs=100, validation_split=0.2,
                    batch_size=64,callbacks=[early_stopping])

Epoch 1/100


I0000 00:00:1743345488.898481  416214 service.cc:146] XLA service 0x736c94013b10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743345488.898885  416214 service.cc:154]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
2025-03-30 20:08:09.631247: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-03-30 20:08:11.627784: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90201


  1/155 ━━━━━━━━━━━━━━━━━━━━ 2:04:36 49s/step - acc: 0.5000 - auc: 0.3379 - false_negatives: 9.0000 - false_positives: 23.0000 - loss: 0.8987 - precision: 0.5660 - recall: 0.7692 - true_negatives: 2.0000 - true_positives: 30.0000

I0000 00:00:1743345522.506631  416214 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


155/155 ━━━━━━━━━━━━━━━━━━━━ 113s 421ms/step - acc: 0.5420 - auc: 0.5503 - false_negatives: 1147.8654 - false_positives: 1091.6923 - loss: 0.7259 - precision: 0.5402 - recall: 0.5315 - true_negatives: 1445.2500 - true_positives: 1338.5256 - val_acc: 0.6377 - val_auc: 0.6708 - val_false_negatives: 617.0000 - val_false_positives: 280.0000 - val_loss: 0.6386 - val_precision: 0.6340 - val_recall: 0.4401 - val_true_negatives: 1094.0000 - val_true_positives: 485.0000
Epoch 2/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 138ms/step - acc: 0.7969 - auc: 0.8861 - false_negatives: 495.5962 - false_positives: 493.8269 - loss: 0.4244 - precision: 0.7924 - recall: 0.8001 - true_negatives: 2034.3782 - true_positives: 1999.5321 - val_acc: 0.6636 - val_auc: 0.7131 - val_false_negatives: 559.0000 - val_false_positives: 274.0000 - val_loss: 0.7366 - val_precision: 0.6646 - val_recall: 0.4927 - val_true_negatives: 1100.0000 - val_true_positives: 543.0000
Epoch 3/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 138ms/step - a

KeyboardInterrupt: 

In [ ]:
test_results = ensemble_model.evaluate([x_test, x_test], y_test, batch_size=64)
print("Test Metrics:", test_results)


50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step - acc: 0.8862 - auc_2: 0.9495 - false_negatives_2: 154.2353 - false_positives_2: 43.8824 - loss: 0.3235 - precision_2: 0.9457 - recall_2: 0.8200 - true_negatives_2: 797.2745 - true_positives_2: 666.2549
Test Metrics: [0.3186386227607727, 0.8824085593223572, 0.8195679783821106, 0.9354605078697205, 0.9499800205230713, 1290.0, 1509.0, 284.0, 89.0]


In [4]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet152V2, EfficientNetB7
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import Accuracy, Precision, Recall, AUC


# Input Shape
img_shape = (128, 128, 3)
inputs = Input(shape=img_shape)

# Load ResNet152V2
resnet = ResNet152V2(weights='imagenet', include_top=False, input_shape=img_shape)
for layer in resnet.layers[:100]:  # Freeze first 100 layers
    layer.trainable = False
resnet_output = GlobalAveragePooling2D()(resnet.output)

# Load EfficientNetB7
efficient = EfficientNetB7(weights='imagenet', include_top=False, input_shape=img_shape)
for layer in efficient.layers[:500]:  # Freeze first 500 layers
    layer.trainable = False
efficient_output = GlobalAveragePooling2D()(efficient.output)

# Concatenate Outputs
merged = Concatenate()([resnet_output, efficient_output])
merged = Dense(256, activation='relu')(merged)
merged = Dropout(0.4)(merged)
merged = Dense(128, activation='relu')(merged)
merged = Dropout(0.25)(merged)
outputs = Dense(1, activation='sigmoid')(merged)

# Create Model
model = Model(inputs=[resnet.input, efficient.input], outputs=outputs)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.metrics import Recall, Precision, AUC, TruePositives, TrueNegatives, FalseNegatives, FalsePositives
from keras.callbacks import EarlyStopping
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
# keras.optimizers.Adam()

model.compile (optimizer=optimizer ,  loss=keras.losses.BinaryCrossentropy() , 
               metrics=['acc',Recall(),Precision(),AUC(),TruePositives(),TrueNegatives(),FalseNegatives(),FalsePositives()])
early_stopping = EarlyStopping(
    monitor='val_acc', 
    patience=15, 
    min_delta=0.001, 
    mode='max',
    restore_best_weights=True
)



2025-03-10 01:28:17.868966: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0


In [5]:
history = model.fit([x_train,x_train], y_train, epochs=100, validation_split=0.2,
                    batch_size=64,callbacks=[early_stopping])

Epoch 1/100


I0000 00:00:1741550392.953456  771883 service.cc:146] XLA service 0x7355400046b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1741550392.953574  771883 service.cc:154]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
2025-03-10 01:29:56.667766: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-03-10 01:30:05.351113: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90201
E0000 00:00:1741550429.156684  771883 gpu_timer.cc:183] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1741550429.328315  771883 gpu_timer.cc:183] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E

154/155 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - acc: 0.6103 - auc: 0.6631 - false_negatives: 945.1883 - false_positives: 800.4805 - loss: 0.6408 - precision: 0.6143 - recall: 0.5726 - true_negatives: 1708.2273 - true_positives: 1506.1039

E0000 00:00:1741550581.128476  771883 gpu_timer.cc:183] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1741550581.319800  771883 gpu_timer.cc:183] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


155/155 ━━━━━━━━━━━━━━━━━━━━ 388s 1s/step - acc: 0.6115 - auc: 0.6648 - false_negatives: 953.5577 - false_positives: 807.2436 - loss: 0.6396 - precision: 0.6156 - recall: 0.5739 - true_negatives: 1733.4935 - true_positives: 1529.0385 - val_acc: 0.6127 - val_auc: 0.7580 - val_false_negatives: 129.0000 - val_false_positives: 830.0000 - val_loss: 1.0012 - val_precision: 0.5397 - val_recall: 0.8829 - val_true_negatives: 544.0000 - val_true_positives: 973.0000
Epoch 2/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 45s 289ms/step - acc: 0.8674 - auc: 0.9396 - false_negatives: 357.5641 - false_positives: 305.0000 - loss: 0.3159 - precision: 0.8729 - recall: 0.8568 - true_negatives: 2240.8718 - true_positives: 2119.8975 - val_acc: 0.6361 - val_auc: 0.7351 - val_false_negatives: 230.0000 - val_false_positives: 671.0000 - val_loss: 1.2049 - val_precision: 0.5651 - val_recall: 0.7913 - val_true_negatives: 703.0000 - val_true_positives: 872.0000
Epoch 3/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 45s 292ms/step - acc: 0.9

In [6]:
test_results = model.evaluate([x_test, x_test], y_test, batch_size=64)
print("Test Metrics:", test_results)


50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 420ms/step - acc: 0.7073 - auc: 0.7710 - false_negatives: 242.1569 - false_positives: 272.0000 - loss: 1.1817 - precision: 0.6981 - recall: 0.7297 - true_negatives: 569.1569 - true_positives: 578.3333
Test Metrics: [1.2230104207992554, 0.6860024929046631, 0.695679783821106, 0.6792804002761841, 0.7523096799850464, 1095.0, 1081.0, 479.0, 517.0]
